# One agent becomes several

Notebook 8 ended with a single agent that could establish facts, choose among
levers, and be refused by a validator when the facts didn't support it. It works.
It also has a ceiling, and the ceiling is *context*.

Every fact that agent ever retrieves stays in one conversation forever. Add
payment ledgers, carrier scans and a written policy to the same loop and three
things go wrong at once: the prompt has to describe every tool to every call, the
transcript fills with evidence that mattered for one step and never again, and
the model's attention is split across four unrelated investigations. Bigger
prompts, worse decisions, higher cost.

The fix isn't a better prompt. It's **structure**: one coordinator that decides
and acts, and several narrow sub-agents that go and find out. Each sub-agent
starts empty, receives exactly the context its question needs, and hands back a
structured finding rather than a transcript.

Three things make that work, and this notebook is about them:

1. **A coordinator** — a hub that routes, aggregates and holds all the authority
   to change anything. The spokes are read-only by construction.
2. **A Task tool** — delegation as an ordinary tool call, so the coordinator
   decides *when* to fan out and the loop you already have runs the sub-agent.
3. **Explicit context passing** — what a sub-agent knows is an argument, not an
   inheritance. Nothing leaks in because there's nothing to leak from.

## First: notebook 8 is now a library

Notebook 8 was 60KB, and by the end most of it was material notebook 6 and 7 had
already taught. Restating it here would make this file unreadable, so it moved
into a `shop/` package next to these notebooks:

| Module | What's in it | Where it was taught |
| --- | --- | --- |
| `shop/data.py` | orders, payments, shipments, policy text, append-only records | nb 7 |
| `shop/policy.py` | `RETURN_WINDOW_DAYS`, `eligibility_facts` | nb 8 |
| `shop/tools.py` | all ten tool schemas + implementations | nb 5–8 |
| `shop/validate.py` | schema layer, then semantic layer, per tool | nb 6, 8 |
| `shop/case.py` | case state, terminal states | nb 7, 8 |
| `shop/runtime.py` | the tool-use loop, stop-reason handling, retry budget | nb 8 |

Two things changed in the move, both because several agents now share this code:

- **`case` is an argument, not a global.** Once more than one agent is in
  flight, "which case is this call about" has to be passed, not assumed.
- **`run_agent(spec, ...)` replaces `send_message()`.** An agent is now a dict —
  prompt, tool list, how it finishes — and one loop runs all of them. That is the
  actual claim of hub-and-spoke: a spoke isn't a different kind of thing, it's
  the same loop pointed at a narrower job.

Four read-only tools are new, because the sub-agents need genuinely different
evidence to gather: `get_payment_events`, `get_refund_status`,
`get_shipment_events`, `search_policy`. None of them decides anything.

Open `shop/` if you want to read it — nothing in there is new material.

In [1]:
import json
import threading

from dotenv import load_dotenv
from anthropic import Anthropic

from shop import execute_tool, find_order, is_closed, new_case, run_agent, tools_for
from shop.validate import semantic_validator

load_dotenv()

client = Anthropic()

# Two models on purpose. Routing is a judgement job — read a messy complaint,
# decide what needs finding out, weigh findings that disagree, then act. Each
# spoke does the opposite: one narrow question, three tools, a fixed output
# shape. That asymmetry is a large part of why hub-and-spoke pays for itself.
COORDINATOR_MODEL = "claude-sonnet-5"
SPOKE_MODEL = "claude-haiku-4-5"

case = new_case()
messages = []

## The spokes: an agent is a dict

Three sub-agents, defined entirely by data. There is no `BillingAgent` class and
no sub-agent loop — `run_agent` from `shop/runtime.py` runs these exactly as it
runs the coordinator.

| Agent | Tools | Cannot |
| --- | --- | --- |
| `order_investigation` | `lookup_order`, `check_return_eligibility`, `get_shipment_events` | decide policy, act |
| `billing_analysis` | `get_payment_events`, `get_refund_status` | move money |
| `policy_review` | `search_policy` | see the order |

**The `tools` list is the first enforcement layer, and it's the strongest one.**
`billing_analysis` can't call `issue_refund` — not because its prompt forbids it,
but because the tool is absent from the request. There is no jailbreak for a
capability the model was never handed. The prompt then says the same thing in
words, so a blocked agent understands *why* it's blocked and reports back instead
of flailing.

Note what `policy_review` doesn't get: any tool that reads the order. It answers
"what does the policy say about X" and cannot be talked into "…and therefore
you should refund A1009", because it has no idea what A1009 is. Narrow tools
produce narrow answers, which is the point of asking it separately.

In [2]:
SPOKE_PREAMBLE = """You are {name}, a sub-agent inside a shop's support system.

You never speak to the customer and you cannot change anything: every tool you
have is read-only. You have been given one objective and a short brief. That
brief is everything you know — you cannot see the customer conversation, the
coordinator's reasoning, or what any other sub-agent is doing right now.

Gather what your objective needs, then call report_finding exactly once. That
call IS your answer; anything you write outside it is discarded unread. Every
fact you report must name the tool it came from. If the brief is missing
something you need, put it in `unanswered` rather than guessing, and never
report anything a tool did not actually return.

{role}"""

SPOKE_ROLES = {
    "order_investigation": """Your job is the order record and its physical history: does the order exist,
what state is it in, is it returnable under the return policy, and did the goods
ever actually reach the customer.

You establish facts. You do not decide whether a return should be granted, what
to tell the customer, or whether to escalate — the coordinator decides that.
Note that calling check_return_eligibility is what puts a return on the record
as checked, so if the objective involves a possible return, check it.""",
    "billing_analysis": """Your job is money: what was authorized, what was captured, what has already
been refunded, and whether anything is disputed. Read the ledger carefully —
two captures for the same amount is a double charge, a capture on a cancelled
order is a charge for goods never sent, and an open chargeback means the funds
are held by the bank and nobody at the shop can move them.

You cannot issue, approve or promise a refund. If money needs to move, say so in
`recommended_next` and let the coordinator act.""",
    "policy_review": """Your job is the written policy, quoted accurately. You have no access to orders,
payments or shipments, and you must not assume anything about a specific order —
if the objective smuggles in a claim about one, treat it as a hypothetical.

Answer in the form "the policy says X", quoting the section, and put anything the
policy does not cover in `unanswered`. A gap in the policy is a finding; an
invented rule is a failure.""",
}

SPOKE_TOOLS = {
    "order_investigation": ["lookup_order", "check_return_eligibility", "get_shipment_events"],
    "billing_analysis": ["get_payment_events", "get_refund_status"],
    "policy_review": ["search_policy"],
}


def spoke_spec(name: str) -> dict:
    """Build a sub-agent spec. REPORT_FINDING is defined in the next cell."""
    return {
        "name": name,
        "model": SPOKE_MODEL,
        "system": SPOKE_PREAMBLE.format(name=name, role=SPOKE_ROLES[name]),
        "tools": tools_for(SPOKE_TOOLS[name]) + [REPORT_FINDING],
        # Calling this tool ends the run and its input becomes the result. A
        # sub-agent that just stops talking has failed, not finished.
        "final_tool": "report_finding",
        # A tighter backstop than the coordinator's: a spoke with three tools and
        # one question has no legitimate reason to need ten rounds.
        "max_iterations": 6,
        # No closing_notes — nobody reads a sub-agent's prose.
    }

## The handoff contract: `report_finding`

A sub-agent that returns a paragraph has handed back a *summary*, and a summary
is where information quietly dies. Which claims were verified? Which tool said
so? What did it fail to find out? Prose can omit all three and still read well.

So the handoff is a tool call with a schema:

```
status            ok | blocked | needs_human
answer            the direct answer to the objective, in one or two sentences
facts             [{claim, source}]  — source is the tool the claim came from
unanswered        what you could not determine, and why
recommended_next  what you think should happen, or null
confidence        low | medium | high
```

Two properties are worth naming.

**Sources are checkable.** The spoke names a tool for each claim, and `run_task`
compares those names against the tools it actually watched the spoke call. A
claim citing a tool that was never invoked is flagged before the coordinator ever
reads it. The model asserts; the harness verifies.

**`recommended_next` is a recommendation, not an instruction.** A sub-agent
writing "issue a refund" is expressing an opinion to an agent that outranks it.
This is exactly the boundary notebook 8 drew between a fact tool and a director,
redrawn one level up — and the reason `issue_refund` isn't in any spoke's tool
list is that opinions shouldn't be self-executing.

In [3]:
REPORT_FINDING = {
    "name": "report_finding",
    "description": (
        "Report what you found and end your task. Call this exactly once, as your "
        "final action. Everything the coordinator will ever know about your work is "
        "in this call — anything you leave out is lost."
    ),
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {
            "status": {
                "type": "string",
                "enum": ["ok", "blocked", "needs_human"],
                "description": (
                    "'ok' if you answered the objective; 'blocked' if you could not "
                    "(missing context, tool errors); 'needs_human' if what you found "
                    "is outside what any assistant may resolve."
                ),
            },
            "answer": {
                "type": "string",
                "description": "One or two sentences answering the objective directly.",
            },
            "facts": {
                "type": "array",
                "description": "The evidence behind your answer. Every claim cites the tool it came from.",
                "items": {
                    "type": "object",
                    "properties": {
                        "claim": {"type": "string", "description": "One specific, checkable fact."},
                        "source": {
                            "type": "string",
                            "description": "The exact name of the tool that returned it, e.g. get_payment_events.",
                        },
                    },
                    "required": ["claim", "source"],
                    "additionalProperties": False,
                },
            },
            "unanswered": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Anything the objective asked that you could not determine, and why.",
            },
            "recommended_next": {
                "type": ["string", "null"],
                "description": (
                    "What you think should happen next, or null. This is advice to the "
                    "coordinator, not an instruction — you are not deciding."
                ),
            },
            "confidence": {"type": "string", "enum": ["low", "medium", "high"]},
        },
        "required": ["status", "answer", "facts", "unanswered", "recommended_next", "confidence"],
        "additionalProperties": False,
    },
}


@semantic_validator("report_finding")
def validate_finding(tool_input: dict, case: dict) -> list[str]:
    """Shape checks the schema can't express. Same two-layer pattern as every
    other tool — the handoff is not exempt from validation just because it's
    internal. It is the *most* load-bearing call a sub-agent makes."""
    errors = []
    status = tool_input.get("status")
    facts = tool_input.get("facts", [])

    if status == "ok" and not facts:
        errors.append(
            "status is 'ok' but you cited no facts. An answer with no evidence "
            "behind it is a guess — either cite what you found or report 'blocked'."
        )
    if status == "blocked" and not tool_input.get("unanswered"):
        errors.append(
            "status is 'blocked' but 'unanswered' is empty. Say what you could not "
            "determine, so the coordinator knows what is still missing."
        )
    if status == "needs_human" and not tool_input.get("recommended_next"):
        errors.append(
            "status is 'needs_human' but recommended_next is null. Say what a human "
            "should look at — this text is what they will read first."
        )
    if len(tool_input.get("answer", "").split()) < 4:
        errors.append("answer is too short to be a real answer to the objective.")

    return errors

## The Task tool: delegation is just a tool call

The coordinator doesn't have a special delegation mechanism. It has a tool called
`Task`, and calling it runs a sub-agent. That keeps the whole design inside
machinery you already have: schema validation, semantic validation, retries,
`tool_result` — a sub-agent run is subject to all of it, because it *is* a tool
call.

The input schema is where explicit context passing lives:

```
agent              which sub-agent
objective          the one question it should answer
context            order_id, customer_statement, notes — what it needs to know
include_findings   which earlier findings to forward, by agent name
```

`context` is not a convenience. **It is the entire world the sub-agent will
wake up in** — `run_task` builds a brand-new `messages` list out of it and
nothing else. The coordinator's conversation, the customer's other complaints,
the previous four tool results: none of it is reachable. If the coordinator
forgets to pass the order ID, the sub-agent genuinely does not have it and says
so in `unanswered` rather than inventing one.

`include_findings` is what makes chaining explicit. The coordinator names which
prior findings travel forward, so passing context is a decision it takes rather
than an accident of history. And the two shapes of decomposition fall out of the
same tool with no extra machinery:

- **Parallel** — two `Task` blocks in one assistant turn. `run_agent` runs them
  concurrently on a thread pool when the spec sets `parallel: True`.
- **Chained** — a `Task` in a later turn, with `include_findings` naming the
  earlier one.

The validator below refuses a chain link that names a finding nobody has produced
yet, which is how "agile decomposition" stays honest: the coordinator can pick
any shape, but it can't claim an input it doesn't have.

In [4]:
TASK_TOOL = {
    "name": "Task",
    "description": (
        "Hand one question to a sub-agent and get a structured finding back. The "
        "sub-agent starts with no memory of this conversation and sees only what you "
        "put in `context` — pass everything it needs. Issue several Task calls in one "
        "turn when the questions are independent; they run in parallel. Use "
        "include_findings when a question depends on what another sub-agent already "
        "reported."
    ),
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {
            "agent": {
                "type": "string",
                "enum": ["order_investigation", "billing_analysis", "policy_review"],
                "description": (
                    "order_investigation: the order record, return eligibility, carrier "
                    "history. billing_analysis: charges, refunds, disputes. "
                    "policy_review: what the written policy says (it cannot see orders)."
                ),
            },
            "objective": {
                "type": "string",
                "description": (
                    "The single question this sub-agent should answer, stated so it can "
                    "be answered with its tools alone — not a plan, and not an instruction to act."
                ),
            },
            "context": {
                "type": "object",
                "description": "Everything the sub-agent gets to know. It sees nothing else.",
                "properties": {
                    "order_id": {
                        "type": ["string", "null"],
                        "description": "The order in question, or null if the question isn't about one.",
                    },
                    "customer_statement": {
                        "type": ["string", "null"],
                        "description": (
                            "What the customer actually said, in their words, if it matters. "
                            "Don't paraphrase away the details."
                        ),
                    },
                    "notes": {
                        "type": ["string", "null"],
                        "description": "Anything else the sub-agent needs that it cannot look up.",
                    },
                },
                "required": ["order_id", "customer_statement", "notes"],
                "additionalProperties": False,
            },
            "include_findings": {
                "type": "array",
                "items": {
                    "type": "string",
                    "enum": ["order_investigation", "billing_analysis", "policy_review"],
                },
                "description": (
                    "Names of sub-agents whose findings should be forwarded. Only agents "
                    "that have already reported on this case."
                ),
            },
        },
        "required": ["agent", "objective", "context", "include_findings"],
        "additionalProperties": False,
    },
}


@semantic_validator("Task")
def validate_task(tool_input: dict, case: dict) -> list[str]:
    """Gate on delegation itself. Nothing here is about what the sub-agent will
    do — it's about whether this task is answerable as written."""
    errors = []
    agent = tool_input.get("agent")
    context = tool_input.get("context") or {}
    order_id = context.get("order_id")

    if len(tool_input.get("objective", "").split()) < 5:
        errors.append(
            "objective is too thin. The sub-agent cannot see this conversation, so a "
            "few words are not enough — state the whole question."
        )

    if order_id is not None and find_order(order_id) is None:
        errors.append(
            f"order_id '{order_id}' doesn't match any order. Don't send a sub-agent "
            "chasing an order that doesn't exist — check it or pass null."
        )

    if agent == "policy_review" and order_id is not None:
        # Not a safety rule, a scoping one: policy_review can't read orders, so an
        # order ID in its brief only invites it to reason about one it can't see.
        errors.append(
            "policy_review has no access to orders, so passing an order_id only "
            "tempts it to guess. Ask the policy question in general terms."
        )

    reported = {f["agent"] for f in case["findings"]}
    missing = [name for name in tool_input.get("include_findings", []) if name not in reported]
    if missing:
        errors.append(
            f"include_findings names {missing}, which has not reported on this case yet. "
            "You cannot forward a finding that doesn't exist — run that task first, or "
            "drop it from the list."
        )
    if agent in tool_input.get("include_findings", []):
        errors.append(f"'{agent}' cannot be sent its own earlier finding.")

    return errors

## `run_task`: where the isolation actually happens

Four lines of this function carry the whole idea:

```python
messages = [{"role": "user", "content": brief}]
```

That's it. A fresh list, built from the `context` object. Not a copy of the
coordinator's messages, not a trimmed version of them — a new conversation whose
first and only user turn is the brief. Isolation isn't enforced here so much as
it's *structural*: there is no code path by which the coordinator's transcript
could reach the sub-agent.

The rest of the function is the plumbing that makes the handoff trustworthy:

- **Per-task tool ledger.** A wrapped dispatch records which tools the spoke
  really called, closed over a local list, so parallel tasks can't confuse each
  other's ledgers.
- **Source verification.** Each fact's `source` is checked against that ledger.
  Unverifiable claims are marked and travel *with* the finding rather than being
  silently dropped — the coordinator should see a weak finding as weak, not have
  it quietly cleaned up.
- **A finding on every path.** A spoke that hit its iteration cap or got refused
  still produces a `blocked` finding. The coordinator must never receive silence,
  because silence is indistinguishable from "nothing was wrong".
- **A lock around the shared case.** Parallel tasks append findings from
  different threads. This is the price of the fan-out and it's worth stating
  plainly rather than getting away with.

In [5]:
# Sub-agents run concurrently and share one case dict. Guard what they append.
CASE_LOCK = threading.Lock()

# Bookkeeping for the inspection cell at the end: what each delegation cost.
TASK_LOG = []


def build_brief(task: dict, case: dict) -> str:
    """The sub-agent's entire universe, as one user message.

    Findings are forwarded as JSON rather than prose deliberately: the receiving
    agent should see the same structure the coordinator saw, including the
    confidence and the unanswered list. Summarising a finding to pass it on is
    how a chain of agents turns facts into rumour.
    """
    context = task["context"]
    lines = [f"OBJECTIVE\n{task['objective']}", "", "CONTEXT"]
    lines += [
        f"- {key}: {context[key]}"
        for key in ("order_id", "customer_statement", "notes")
        if context.get(key)
    ] or ["- (none given)"]

    forwarded = [f for f in case["findings"] if f["agent"] in task["include_findings"]]
    if forwarded:
        lines += ["", "FINDINGS FROM OTHER SUB-AGENTS", json.dumps(forwarded, indent=2)]

    lines += ["", "Answer the objective with your tools, then call report_finding."]
    return "\n".join(lines)


def verify_sources(finding: dict, tools_called: list[str]) -> dict:
    """Check each fact's cited tool against the tools this spoke actually ran.

    The model asserts where a fact came from; only the harness knows. A claim
    citing a tool that was never called is not necessarily false — but it is
    unverified, and the difference should reach the coordinator intact.
    """
    unverified = [
        fact["claim"] for fact in finding.get("facts", [])
        if fact.get("source") not in tools_called
    ]
    return {**finding, "tools_called": tools_called, "unverified_claims": unverified}


def run_task(task: dict, case: dict) -> tuple[str, bool]:
    """Run one sub-agent in its own context and return its finding as a tool result."""
    agent = task["agent"]
    tools_called: list[str] = []

    def spoke_dispatch(name, tool_input, case):
        if name == "report_finding":
            # The handoff has no implementation and needs none: the call itself is
            # the result. It only has to succeed, so the loop treats it as the
            # final tool. It stays out of the ledger too — citing report_finding
            # as the source of a fact is not evidence of anything.
            return json.dumps({"received": True}), False

        # Closed over this task's own list, so parallel spokes don't share a ledger.
        tools_called.append(name)
        return execute_tool(name, tool_input, case)

    brief = build_brief(task, case)
    # THE isolation line: a brand-new conversation containing only the brief.
    spoke_messages = [{"role": "user", "content": brief}]

    print(f"[Task → {agent}] {task['objective']}")
    outcome = run_agent(
        client, spoke_spec(agent), spoke_messages, case, dispatch=spoke_dispatch
    )

    finding = outcome["result"]
    if finding is None:
        # The spoke stopped without reporting — ran out of iterations, was refused,
        # or simply replied in prose. The coordinator gets a finding regardless.
        finding = {
            "status": "blocked",
            "answer": f"{agent} ended without reporting a finding ({outcome['stop']}).",
            "facts": [],
            "unanswered": [task["objective"]],
            "recommended_next": "Re-task with more context, or handle this without it.",
            "confidence": "low",
        }

    finding = verify_sources({"agent": agent, "objective": task["objective"], **finding}, tools_called)

    with CASE_LOCK:
        case["findings"].append(finding)
        TASK_LOG.append({
            "agent": agent,
            "objective": task["objective"],
            "brief_chars": len(brief),
            "tools_called": tools_called,
            "iterations": outcome["iterations"],
            "usage": outcome["usage"],
        })

    return json.dumps(finding), False

## The coordinator

The hub. It talks to the customer, decides what needs finding out, and is the
only agent in the system that can change anything.

Its tool list is the design in one line:

```python
tools_for(["lookup_order", "create_return_authorization",
           "issue_refund", "escalate_to_human"]) + [TASK_TOOL]
```

**All write authority at the hub, all evidence-gathering at the spokes.** Three
consequences worth sitting with:

*It cannot investigate.* No `get_payment_events`, no `get_shipment_events`, no
`search_policy`. To learn anything beyond the bare order record it must delegate.
That's a constraint, and it's the one that makes the architecture real rather
than decorative — a coordinator with every tool would just do everything itself.

*It cannot check eligibility either.* `check_return_eligibility` belongs to
`order_investigation`. But the validator on `create_return_authorization` still
requires an eligibility record on the case — so the coordinator can only
authorize a return **after** a sub-agent has actually checked one. Notice how
that works: the *facts* travel through the shared `case`, the *narrative* travels
through findings. Trusted state is shared; context is not. Those are different
channels on purpose, and conflating them is how multi-agent systems start
believing their own summaries.

*One `Task` in a later turn is a chain; two in one turn is a fan-out.* Hence
`"parallel": True` in the spec.

In [6]:
COORDINATOR_SYSTEM = """You are the coordinator for an online shop's support desk. You talk to the
customer, you decide what needs to be found out, and you are the only part of
this system that can change anything.

## Return policy

- Items can be returned within 30 days of the order date.
- Only orders that were actually delivered or shipped can be returned. A
  cancelled or still-processing order was never delivered, so there is nothing
  to send back.
- An approved return gets an RMA number first, then a refund for the full order
  total. The refund is never issued without an RMA.
- Billing disputes — a customer charged for something they didn't receive, or
  charged twice — are not returns. A human handles those.

## Your team

You cannot investigate anything yourself. Three sub-agents can, and you reach
them with the Task tool:

- **order_investigation** — the order record, return eligibility, carrier
  history. Use it for anything about the order or whether the goods arrived.
- **billing_analysis** — what was charged, what was refunded, whether anything
  is disputed. Use it whenever money is in question.
- **policy_review** — quotes the written policy. It cannot see orders, so ask it
  general questions.

Each sub-agent starts with no memory of this conversation and sees only what you
put in `context`. Pass the order ID and the customer's own words; a sub-agent
given a vague brief will correctly tell you it couldn't answer.

Send several Task calls in one turn when the questions don't depend on each
other — they run at the same time. Chain a later Task with include_findings when
it does depend on an earlier answer. Don't delegate what you can already answer
from a finding you have, and don't re-task an agent to confirm something it
already told you.

Sub-agents advise; you decide. A finding that says "issue a refund" is an
opinion from something that cannot issue refunds.

## Acting

Only you can create_return_authorization, issue_refund and escalate_to_human.
A return can only be authorized once order_investigation has actually checked
eligibility for that order — the tool will refuse otherwise, and that is
deliberate, not a bug to work around.

Escalate when a human is genuinely needed: billing disputes, an open chargeback,
a policy exception you can't grant, an angry customer whose problem you can't
solve. Everything your sub-agents found is attached to the ticket automatically,
so write the summary for a human who will read it cold.

## Hard rules

1. Never tell the customer something has happened until the tool returned
   ok: true. If a tool fails or is rejected, say plainly that it didn't go
   through — never describe a failed action as if it succeeded.
2. Never state a fact no finding gave you. If a finding lists something under
   `unverified_claims` or reports low confidence, don't repeat it as certain.
3. Don't invent policy. If the rules above don't cover it, ask policy_review or
   escalate.
4. One resolution per case: either the return is authorized and refunded, or the
   case goes to a human.

Be warm and direct. Keep replies short — this is a support chat, not a letter.
The customer never hears about sub-agents, tasks or findings; they hear an answer.
"""

CLOSED_CASE_NOTE = """CLOSING MESSAGE. This case is now closed and this is the last thing you will
say — the conversation ends here and you will not see a reply. Write a short
sign-off that confirms what was done, quotes the RMA, refund, or ticket number,
and says what happens next and roughly when. Do not offer further help, do not
ask whether there's anything else, and do not ask any question at all.
"""

UNRESOLVED_NOTE = """CLOSING MESSAGE. You could not complete this request, and no action was taken —
no return was authorized, no refund was issued, no ticket was filed. Say so
honestly without inventing a reason, and never imply anything happened. Tell the
customer in one or two sentences what you need from them to move forward.
"""

COORDINATOR = {
    "name": "coordinator",
    "model": COORDINATOR_MODEL,
    "system": COORDINATOR_SYSTEM,
    # Task sits alongside the levers: delegating and acting are the same kind of
    # move as far as the loop is concerned.
    "tools": tools_for([
        "lookup_order",
        "create_return_authorization",
        "issue_refund",
        "escalate_to_human",
    ]) + [TASK_TOOL],
    # Several Task calls in one turn run concurrently.
    "parallel": True,
    "max_tokens": 1500,
    "closing_notes": {"closed": CLOSED_CASE_NOTE, "unresolved": UNRESOLVED_NOTE},
}


def coordinator_dispatch(tool_name: str, tool_input: dict, case: dict) -> tuple[str, bool]:
    """One extra branch on notebook 8's dispatcher: Task runs a sub-agent.

    This is the seam notebook 10 wraps with PreToolUse/PostToolUse hooks — which
    is why delegation lives here, in dispatch, rather than inside the loop.
    """
    if tool_name == "Task":
        return run_task(tool_input, case)

    return execute_tool(tool_name, tool_input, case)


def send_message(messages: list) -> str:
    """One customer turn, start to finish."""
    outcome = run_agent(
        client,
        COORDINATOR,
        messages,
        case,
        dispatch=coordinator_dispatch,
        stop_when=lambda c: f"case closed ({c['state']})" if is_closed(c) else None,
    )
    return outcome["text"]

## Chat

The fixtures were built so the sub-agents disagree in useful ways. Worth trying:

- **"A1006 never turned up and I think you charged me twice"** — the fan-out
  case. Two independent questions (did it ship? what was charged?) that should go
  out in one turn. The carrier record says `returned_to_sender`; the ledger shows
  two captures of $149.00. Neither sub-agent can see the other's half.
- **"I want to return A1005 — and I was charged for it"** — cancelled *and*
  outside the window, but really a billing dispute. Watch the coordinator refuse
  to authorize a return and escalate instead, with both findings attached.
- **"Refund A1009, it's been weeks"** — delivered fine, but the bank opened a
  chargeback. The money is held; the policy says a human. This is the one where
  `billing_analysis` finding the chargeback should stop the refund, and where a
  `policy_review` task earns its place.
- **"A1001 arrived defective, I'd like my money back"** — the happy path, and
  the interesting mechanical detail: the coordinator can't check eligibility
  itself, so the RMA is only possible after `order_investigation` has run.
- **"A1002 hasn't moved in weeks"** — 20 days without a carrier scan. Not a
  return at all; there's nothing to send back.

Watch for the failure modes too — over-delegation (a Task to confirm something
already in a finding), and under-context (a brief with no `order_id`, and a
sub-agent correctly reporting that it couldn't answer).

Run the inspection cell afterwards for the cost and isolation numbers.

In [7]:
while True:
    try:
        user_input = input("You: ")
        print(f"User: {user_input}")
    except (EOFError, KeyboardInterrupt):
        break

    if not user_input.strip() or user_input.lower() in ("quit", "exit"):
        break

    messages.append({"role": "user", "content": user_input})
    reply = send_message(messages)
    print(f"Assistant: {reply}")

    if is_closed(case):
        print(f"\n--- case closed ({case['state']}): {case['actions']} ---")
        break

User: Hello I have problem with my A1006 order
Assistant: I'd like to help! Could you tell me a bit more about what's going on with order A1006 — is it about a return, a charge, or something else?
User: It is never turned up and I was charged twice
[coordinator] → Task({'agent': 'order_investigation', 'objective': 'Check the status and delivery/carrier history of order A1006 — was it delivered, shipped, lost, or still processing?', 'context': {'order_id': 'A1006', 'customer_statement': 'It is never turned up and I was charged twice', 'notes': 'Customer claims non-delivery. Need order status and carrier history to assess.'}, 'include_findings': []})
[coordinator] → Task({'agent': 'billing_analysis', 'objective': "Check whether order A1006 was charged twice, and whether there's an existing dispute or refund on record.", 'context': {'order_id': 'A1006', 'customer_statement': 'It is never turned up and I was charged twice', 'notes': 'Customer claims double charge on this order.'}, 'include

## What it cost, and what stayed separate

The claim at the top of this notebook was that structure buys you context. Here
is where that stops being an argument and becomes two numbers.

`brief_chars` is how much context each sub-agent was given. Compare it with the
coordinator's input tokens, which grow with every turn of the customer
conversation *and* every finding returned. The spokes stay flat no matter how
long the case runs — that's the property you bought, and it's why this scales to
a tenth sub-agent while a single-loop agent does not.

The second table is the audit trail. Every claim in the case, which sub-agent
made it, which tool it came from, and whether the harness could verify it. For a
case that ended up escalated, this is also what a human now receives — the
escalation ticket carries the findings, so the person picking it up starts where
the machine stopped instead of re-asking everything from scratch.

In [8]:
print("=== delegation ===")
for entry in TASK_LOG:
    usage = entry["usage"]
    print(
        f"{entry['agent']:<22} brief={entry['brief_chars']:>5} chars  "
        f"tools={','.join(entry['tools_called']) or '-':<45} "
        f"rounds={entry['iterations']}  "
        f"tokens={usage['input_tokens']}in/{usage['output_tokens']}out"
    )

print("\n=== findings ===")
for finding in case["findings"]:
    flag = f"  UNVERIFIED: {finding['unverified_claims']}" if finding["unverified_claims"] else ""
    print(f"\n[{finding['agent']}] {finding['status']} ({finding['confidence']} confidence){flag}")
    print(f"  {finding['answer']}")
    for fact in finding["facts"]:
        print(f"    - {fact['claim']}  ({fact['source']})")
    for gap in finding["unanswered"]:
        print(f"    ? {gap}")
    if finding["recommended_next"]:
        print(f"    -> suggested: {finding['recommended_next']}")

print(f"\n=== case ===\nstate: {case['state']}\nactions: {case['actions']}")

=== delegation ===
billing_analysis       brief=  329 chars  tools=get_payment_events,get_refund_status          rounds=2  tokens=3663in/737out
order_investigation    brief=  374 chars  tools=lookup_order,get_shipment_events              rounds=2  tokens=3951in/710out

=== findings ===

[billing_analysis] ok (high confidence)
  Order A1006 was indeed charged twice ($298 total captured instead of $149), but the customer has already been fully refunded $149 as of August 9, 2026. There is no open dispute on record.
    - Two separate captures totaling $298.00 were processed on order A1006: one for $149.00 on 2026-07-20 and a second for $149.00 on 2026-07-21, marked as a retry of an already-succeeded capture.  (get_payment_events)
    - No chargeback is currently open on this order.  (get_payment_events)
    - A refund of $149.00 (REF-B4614395) has already been processed and recorded in the shop's refund system as of 2026-08-09.  (get_refund_status)
    - The order is marked as fully refun

## Where this leaves us

Closed:

- **Context isolation.** A sub-agent's world is an argument. The coordinator's
  transcript cannot reach it, so it cannot be confused by it.
- **Capability isolation.** A spoke's tool list is its authority. `issue_refund`
  is unreachable from `billing_analysis` in the strongest sense available.
- **Structured handoff.** Findings are schema'd, validated, source-checked, and
  they travel intact all the way onto the escalation ticket a human reads.

Still open, and the subject of notebook 10:

1. **Enforcement is scattered.** Three different mechanisms hold the line right
   now: tool lists, schema validators, semantic validators. None of them can
   express "no sub-agent may ever write to disk" as *one* rule — it's currently
   true only because every spoke's tool list happens to omit the write tools. A
   fourth spoke added carelessly breaks it silently.
2. **Nothing observes.** `TASK_LOG` is a print statement, not an audit trail.
   There's no record of what was attempted and denied, which is the half you
   actually need when something goes wrong.
3. **No cross-cutting policy.** "Freeze all writes once the case is closed",
   "redact card numbers before they enter any context", "never let a sub-agent
   call the same tool twice with the same input" — none of these belong in a
   per-tool validator, because none of them are about a particular tool's
   arguments. They're about the caller and the case.

That's the shape of a **hook**: `PreToolUse` can deny or rewrite a call before it
runs, `PostToolUse` can record or redact what it returned — and neither knows
what tool it's looking at. The seam is already built: `dispatch` is a parameter,
and notebook 10 passes a wrapped one.